# Convertible bonds

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_across_asset_classes.ipynb`. **`convertible_bond`** combines **rates + credit + equity vol**. In this build, `credit_curve_id` must reference a **discount** (risky curve), not a hazard curve.


## Concept

- **Conversion ratio / price:** shares per bond vs effective conversion strike.
- **Conversion premium:** equity upside vs straight bond floor.
- **Bond floor:** straight debt component before conversion optionality.


In [ ]:
import json
from datetime import date

from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import DiscountCurve, MarketContext, VolSurface

print("Imports OK (convertibles).")

## Instrument JSON

**Conversion terms:** **Conversion ratio is 25 shares per $1,000 par** (i.e. **conversion price = $40** per share before anti-dilution adjustments).


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

cb = json.dumps(
    {
        "type": "convertible_bond",
        "spec": {
            "id": "CB-TECH-5Y",
            "notional": {"amount": "1000000", "currency": "USD"},
            "issue_date": "2024-01-15",
            "maturity": "2029-01-15",
            "discount_curve_id": "USD-IG",
            "credit_curve_id": "USD-CREDIT-BBB",
            "conversion": {
                "ratio": 25.0,
                "policy": "Voluntary",
                "anti_dilution": "None",
                "dividend_adjustment": "None",
            },
            "underlying_equity_id": "TECH",
            "fixed_coupon": {
                "coupon_type": "Cash",
                "rate": 0.02,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "end_of_month": False,
                "payment_lag_days": 0,
                "stub": "None",
            },
            "attributes": {"tags": [], "meta": {}},
            "pricing_overrides": {},
        },
    }
)

validate_instrument_json(cb)
print("convertible JSON OK")


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85)],
    day_count="act_365f",
)
ig = DiscountCurve(
    "USD-IG",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.965), (5.0, 0.80)],
    day_count="act_365f",
)
risky = DiscountCurve(
    "USD-CREDIT-BBB",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.94), (5.0, 0.72)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois).insert(ig).insert(risky)
mc.insert(VolSurface("TECH-VOL", [0.25, 0.5, 1.0, 2.0], [20.0, 40.0, 60.0], [0.35] * (4 * 3)))
mc.insert_price("TECH", 40.0, "USD")
market_json = mc.to_json()
print("surface:", mc.get_surface("TECH-VOL").id)

## Pricing


In [ ]:
out = price_instrument_with_metrics(cb, market_json, AS_OF_STR, model="discounting")
print(ValuationResult.from_json(out))


## Metrics


In [ ]:
metrics = ["conversion01", "delta", "vega"]
out = price_instrument_with_metrics(
    cb,
    market_json,
    AS_OF_STR,
    model="discounting",
    metrics=metrics,
)
vr = ValuationResult.from_json(out)
for metric in metrics:
    value = vr.get_metric(metric)
    if value is not None:
        print(metric, ":", value)


## Takeaways

- **Risky discount curve** as `credit_curve_id` matches the current loader expectations.
- **Equity spot + vol surface** IDs must exist under `prices` / `surfaces`.
- Conversion metrics decompose **fixed-income floor** vs **equity optionality**.
